# 07 — Selective Random-Walk Hybrid

This notebook keeps Random Walk as the default forecast and only applies an ML correction when historical validation supports it.

It uses leakage-safe previous-year tuning, volatility-normalized targets, signal gating, clipping, shrinkage, and multiple ML models.

Core metrics: RMSE, MAE, Forecast Skill vs RW, Theil's U, DM p-value.


In [ ]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",180)
pd.set_option("display.width",220)
pd.set_option("display.float_format",lambda x:f"{x:,.6f}")
CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
VALIDATION_YEARS=[2019,2020,2021,2022,2023]
FEATURE_FAMILIES=["TT_OWN","TT_CROSS","FULL_PIT"]
TRAIN_WINDOWS=["EXPANDING","RECENT_3Y","RECENT_5Y"]
TARGET_MODES=["RAW_CHANGE","VOL_NORMALIZED"]
LAMBDA_GRID=[0.10,0.25,0.50,0.75,1.00]
GATE_QUANTILES=[0.00,0.25,0.50,0.75]
CLIP_MULTIPLIERS=[1.0,1.5,2.0]
RANDOM_STATE=42
print("Configuration loaded.")

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        feat=root/"data"/"processed"/"02_feature_sets.json"
        rwf=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        if dev.exists() and feat.exists() and rwf.exists(): return root.resolve()
    raise FileNotFoundError("Run Scripts 02 and 04 first.")
ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
FEATURE_PATH=ROOT/"data"/"processed"/"02_feature_sets.json"
RW_PATH=ROOT/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
TABLE_DIR=ROOT/"outputs"/"tables"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR.mkdir(parents=True,exist_ok=True); PRED_DIR.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DEV_PATH,parse_dates=["Date"])
for h in HORIZONS:
    c=f"Target_Date_H{h}"
    if c in df.columns: df[c]=pd.to_datetime(df[c],errors="coerce")
rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
with open(FEATURE_PATH,"r",encoding="utf-8") as f: FEATURE_SETS=json.load(f)
df=df.sort_values("Date").reset_index(drop=True); rw=rw.sort_values(["Currency","Horizon","Forecast_Origin"]).reset_index(drop=True)
assert df["Date"].max()<pd.Timestamp("2024-01-01")
print("Development rows:",len(df)); print("PASS — holdout not loaded.")

In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan
def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan
def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def dm_test(y,model,rw_pred,horizon=1):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    mask=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred); y=y[mask]; model=model[mask]; rw_pred=rw_pred[mask]; n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2; lag=max(0,int(horizon)-1); lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        lrv+=2*(1-k/(lag+1))*np.cov(d[k:],d[:-k],ddof=1)[0,1]
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n); p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)

In [ ]:
def build_model(key):
    if key=="RIDGE": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",Ridge(alpha=10.0))])
    if key=="HUBER": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",HuberRegressor(epsilon=1.35,alpha=0.001,max_iter=500))])
    if key=="ELASTIC_NET": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",ElasticNet(alpha=0.001,l1_ratio=0.20,max_iter=5000,random_state=RANDOM_STATE))])
    if key=="EXTRA_TREES": return Pipeline([("imp",SimpleImputer(strategy="median")),("m",ExtraTreesRegressor(n_estimators=120,max_depth=6,min_samples_leaf=10,max_features=0.70,random_state=RANDOM_STATE,n_jobs=-1))])
    if key=="HIST_GB": return Pipeline([("imp",SimpleImputer(strategy="median")),("m",HistGradientBoostingRegressor(learning_rate=0.03,max_iter=140,max_leaf_nodes=15,l2_regularization=3.0,random_state=RANDOM_STATE))])
    raise ValueError(key)
MODEL_KEYS=["RIDGE","HUBER","ELASTIC_NET","EXTRA_TREES","HIST_GB"]
print(MODEL_KEYS)

In [ ]:
def get_scale(frame,cur):
    c=f"{cur}_return_std14"
    if c in frame.columns: vol=pd.to_numeric(frame[c],errors="coerce")
    else:
        ret=np.log(frame[f"{cur}_mid"]/frame[f"{cur}_mid"].shift(1)); vol=ret.rolling(14,min_periods=7).std()
    level=pd.to_numeric(frame[f"{cur}_mid"],errors="coerce"); scale=(vol*level).replace([np.inf,-np.inf],np.nan)
    positive=scale[scale>0]; fallback=float(positive.median()) if len(positive) else 1.0
    return scale.fillna(fallback).clip(lower=max(fallback*0.05,1e-8))
def apply_window(frame,cutoff,window):
    out=frame[frame["Date"]<cutoff].copy()
    if window=="RECENT_3Y": out=out[out["Date"]>=cutoff-pd.DateOffset(years=3)].copy()
    elif window=="RECENT_5Y": out=out[out["Date"]>=cutoff-pd.DateOffset(years=5)].copy()
    return out
def make_batch(cur,h,family,window,target_mode,val_year):
    features=[c for c in FEATURE_SETS[cur][family] if c in df.columns]; tdate=f"Target_Date_H{h}"; change=f"{cur}_change_H{h}"
    start=pd.Timestamp(f"{val_year}-01-01"); end=pd.Timestamp(f"{val_year+1}-01-01"); base=df.copy(); base["_scale"]=get_scale(base,cur)
    base["_target"]=base[change] if target_mode=="RAW_CHANGE" else base[change]/base["_scale"]
    train=apply_window(base,start,window); train=train[train["_target"].notna()&train[tdate].notna()&(train[tdate]<start)].copy()
    val=base[(base["Date"]>=start)&(base["Date"]<end)&base["_target"].notna()&base[tdate].notna()].copy()
    return train,val,features
def raw_change(pred,val,target_mode): return np.asarray(pred,float) if target_mode=="RAW_CHANGE" else np.asarray(pred,float)*val["_scale"].to_numpy(float)

In [ ]:
def evaluate_inner(cur,h,outer_year,family,window,target_mode,model_key):
    inner_year=outer_year-1; train,val,features=make_batch(cur,h,family,window,target_mode,inner_year)
    if len(train)<200 or len(val)<30 or not features: return None
    model=build_model(model_key)
    try:
        model.fit(train[features],train["_target"]); pred=model.predict(val[features])
    except Exception: return None
    rc=raw_change(pred,val,target_mode); current=val[f"{cur}_mid"].to_numpy(float); actual=val[f"{cur}_target_H{h}"].to_numpy(float)
    ch=np.abs(train[f"{cur}_change_H{h}"].to_numpy(float)); ch=ch[np.isfinite(ch)]
    if len(ch)==0: return None
    return {"Actual":actual,"RW":current,"Raw_Change":rc,"Base_Clip":float(np.quantile(ch,0.95))}
def tune_task(cur,h,outer_year):
    rows=[]
    for family in FEATURE_FAMILIES:
      for window in TRAIN_WINDOWS:
       for target_mode in TARGET_MODES:
        for model_key in MODEL_KEYS:
         res=evaluate_inner(cur,h,outer_year,family,window,target_mode,model_key)
         if res is None: continue
         sig=np.abs(res["Raw_Change"])
         for clip_mult in CLIP_MULTIPLIERS:
          clipped=np.clip(res["Raw_Change"],-res["Base_Clip"]*clip_mult,res["Base_Clip"]*clip_mult)
          for gate_q in GATE_QUANTILES:
           threshold=float(np.quantile(sig,gate_q)); gate=(sig>=threshold).astype(float)
           for lam in LAMBDA_GRID:
            pred=res["RW"]+lam*gate*clipped
            rows.append({"Feature_Family":family,"Train_Window":window,"Target_Mode":target_mode,"Model":model_key,"Clip_Multiplier":clip_mult,"Gate_Quantile":gate_q,"Gate_Threshold":threshold,"Lambda":lam,"Inner_RMSE":rmse(res["Actual"],pred)})
    tab=pd.DataFrame(rows)
    if tab.empty: return None,tab
    b=tab.sort_values("Inner_RMSE").iloc[0]
    return {"Feature_Family":b["Feature_Family"],"Train_Window":b["Train_Window"],"Target_Mode":b["Target_Mode"],"Model":b["Model"],"Clip_Multiplier":float(b["Clip_Multiplier"]),"Gate_Quantile":float(b["Gate_Quantile"]),"Gate_Threshold":float(b["Gate_Threshold"]),"Lambda":float(b["Lambda"]),"Inner_RMSE":float(b["Inner_RMSE"])},tab

In [ ]:
def predict_outer(cur,h,outer_year,sel):
    train,outer,features=make_batch(cur,h,sel["Feature_Family"],sel["Train_Window"],sel["Target_Mode"],outer_year)
    rw_task=rw[(rw["Currency"]==cur)&(rw["Horizon"]==h)&(rw["Forecast_Origin"].dt.year==outer_year)][["Forecast_Origin","Target_Date","Actual","RW_Prediction"]].copy()
    outer=outer.merge(rw_task,left_on="Date",right_on="Forecast_Origin",how="inner")
    if outer.empty: return pd.DataFrame()
    model=build_model(sel["Model"]); model.fit(train[features],train["_target"]); pred=model.predict(outer[features]); rc=raw_change(pred,outer,sel["Target_Mode"])
    ch=np.abs(train[f"{cur}_change_H{h}"].to_numpy(float)); ch=ch[np.isfinite(ch)]; base_clip=float(np.quantile(ch,0.95))
    clipped=np.clip(rc,-base_clip*sel["Clip_Multiplier"],base_clip*sel["Clip_Multiplier"]); gate=(np.abs(rc)>=sel["Gate_Threshold"]).astype(float)
    rwv=outer["RW_Prediction"].to_numpy(float); final=rwv+sel["Lambda"]*gate*clipped
    return pd.DataFrame({"Currency":cur,"Horizon":h,"Validation_Year":outer_year,"Forecast_Origin":outer["Forecast_Origin"],"Target_Date":outer["Target_Date"],"Actual":outer["Actual"],"RW_Prediction":outer["RW_Prediction"],"Hybrid_Prediction":final,"Gate":gate,"Feature_Family":sel["Feature_Family"],"Train_Window":sel["Train_Window"],"Target_Mode":sel["Target_Mode"],"Selected_Model":sel["Model"],"Lambda":sel["Lambda"],"Gate_Quantile":sel["Gate_Quantile"],"Clip_Multiplier":sel["Clip_Multiplier"]})

In [ ]:
all_preds=[]; selections=[]; cand_tabs=[]; task=0; total=60; start_all=time.time()
for cur in CURRENCIES:
 for h in HORIZONS:
  for year in VALIDATION_YEARS:
   task+=1; t0=time.time(); print(f"[{task:02d}/{total}] {cur} H{h} — {year}",flush=True)
   sel,tab=tune_task(cur,h,year)
   if sel is None: sel={"Feature_Family":"TT_OWN","Train_Window":"EXPANDING","Target_Mode":"RAW_CHANGE","Model":"RIDGE","Clip_Multiplier":1.0,"Gate_Quantile":0.75,"Gate_Threshold":np.inf,"Lambda":0.0,"Inner_RMSE":np.nan}
   if not tab.empty:
    temp=tab.copy(); temp["Currency"]=cur; temp["Horizon"]=h; temp["Outer_Year"]=year; cand_tabs.append(temp)
   all_preds.append(predict_outer(cur,h,year,sel)); sec=time.time()-t0
   selections.append({"Currency":cur,"Horizon":h,"Outer_Year":year,"Inner_Validation_Year":year-1,**sel,"Task_Seconds":sec})
   print(f"   {sel['Model']} | {sel['Feature_Family']} | {sel['Target_Mode']} | gate={sel['Gate_Quantile']:.2f} | lambda={sel['Lambda']:.2f} | {sec:.1f}s",flush=True)
hybrid_predictions=pd.concat(all_preds,ignore_index=True); hybrid_selections=pd.DataFrame(selections)
candidate_results=pd.concat(cand_tabs,ignore_index=True) if cand_tabs else pd.DataFrame()
print(f"Finished in {(time.time()-start_all)/60:.2f} minutes.")

In [ ]:
yearly=[]
for (cur,h,year),g in hybrid_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    mr=rmse(g["Actual"],g["Hybrid_Prediction"]); ma=mae(g["Actual"],g["Hybrid_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["Hybrid_Prediction"],g["RW_Prediction"],h)
    yearly.append({"Currency":cur,"Horizon":h,"Validation_Year":year,"Observations":len(g),"RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"]); display(yearly_results); yearly_results.to_csv(TABLE_DIR/"07_hybrid_yearly_results.csv",index=False)

In [ ]:
pooled=[]
for (cur,h),g in hybrid_predictions.groupby(["Currency","Horizon"]):
    mr=rmse(g["Actual"],g["Hybrid_Prediction"]); ma=mae(g["Actual"],g["Hybrid_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["Hybrid_Prediction"],g["RW_Prediction"],h)
    pooled.append({"Currency":cur,"Horizon":h,"Observations":len(g),"RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})
pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"]); display(pooled_results); pooled_results.to_csv(TABLE_DIR/"07_hybrid_pooled_results.csv",index=False)
wins=pooled_results.assign(Beats_RW=pooled_results["Forecast_Skill_vs_RW"]>0,DM_Significant_5pct=pooled_results["DM_pvalue_vs_RW"]<0.05); display(wins); print("Tasks beating RW:",int(wins["Beats_RW"].sum()),"/",len(wins))

In [ ]:
hybrid_predictions.to_csv(PRED_DIR/"07_hybrid_predictions_2019_2023.csv",index=False); hybrid_selections.to_csv(TABLE_DIR/"07_hybrid_selections.csv",index=False)
if not candidate_results.empty: candidate_results.to_csv(TABLE_DIR/"07_hybrid_inner_candidates.csv",index=False)
gate_summary=hybrid_predictions.groupby(["Currency","Horizon"]).agg(Gate_On_Percent=("Gate",lambda s:100*np.mean(s)),Observations=("Gate","size")).reset_index(); display(hybrid_selections); display(gate_summary); gate_summary.to_csv(TABLE_DIR/"07_gate_usage_summary.csv",index=False)

In [ ]:
rw_keys=set(zip(rw["Currency"],rw["Horizon"],rw["Forecast_Origin"])); hybrid_keys=set(zip(hybrid_predictions["Currency"],hybrid_predictions["Horizon"],hybrid_predictions["Forecast_Origin"]))
quality_gate=pd.DataFrame([{ "Area":"Holdout isolation","Status":"PASS","Evidence":"Only pre-2024 data used"},{"Area":"Frozen RW","Status":"PASS","Evidence":"Script 04 reused"},{"Area":"Inner tuning","Status":"PASS","Evidence":"Previous year only"},{"Area":"Target overlap","Status":"PASS","Evidence":"Training targets precede validation start"},{"Area":"Signal gating","Status":"PASS","Evidence":"Gate selected on inner year"},{"Area":"Clipping","Status":"PASS","Evidence":"Training distribution only"},{"Area":"Matched origins","Status":"PASS" if rw_keys==hybrid_keys else "REVIEW","Evidence":f"RW={len(rw_keys)}, Hybrid={len(hybrid_keys)}"}]); display(quality_gate); quality_gate.to_csv(TABLE_DIR/"07_quality_gate.csv",index=False)

In [ ]:
print("="*80); print("SCRIPT 07 — SELECTIVE RW HYBRID COMPLETE"); print("="*80); display(pooled_results); print("\nTasks beating RW:",int((pooled_results["Forecast_Skill_vs_RW"]>0).sum()),"/",len(pooled_results)); print("Forecast Skill > 0 and Theil U < 1 = lower RMSE than RW."); print("DM p < 0.05 strengthens a positive result."); print("2024–2025 remains untouched.")